
# Slead 3b — **Fast Station-Only Forward Model** (keeps your variables)

This mirrors the 3a fast path but for **model 3b**.  
It preserves your variable names (`rectangles`, `patches`, `stations`, `station_utm`, `Ustat`, `predict_range_change`) and tries to load **`model3b.h5` first**, falling back to `model3a.h5` if needed.


In [1]:
import sys, os
sys.path.insert(0, os.path.join('..', 'src'))

import os, h5py, numpy as np
import utm
from pyproj import Proj, transform

# Okada impl
try:
    from VSM_forward import okada as OKADA  # your implementation
    print("Using VSM_forward.okada")
except Exception:
    OKADA = None
    print("WARNING: Using stub okada; replace with VSM_forward.okada for real results")


Using VSM_forward.okada


In [2]:

def _okada_stub(x, y, xtlc, ytlc, dtlc, length, width, strike, dip,
                ss, ds, slip, rake, nu):
    # Simple smooth field for testing (not physical)
    xv = x - xtlc; yv = y - ytlc
    r2 = xv*xv + yv*yv + (0.3*length)**2
    fac = slip / (r2 + 1e-6)
    return fac * xv, fac * yv, fac * (xv*yv / (r2+1e-6))

if OKADA is None:
    OKADA = _okada_stub


In [3]:

# 1) Load everything from the .h5 (prefer model3b.h5; fallback to model3a.h5)
h5_candidates = ["../Data/model3b.h5", "../Data/model3a.h5"]
h5_used = None
for cand in h5_candidates:
    if os.path.exists(cand):
        h5_used = cand
        break
if h5_used is None:
    raise FileNotFoundError("Neither model3b.h5 nor model3a.h5 found in this directory.")

print("Loading:", h5_used)
with h5py.File(h5_used,'r') as f:
    # required datasets
    m          = f['m'][:]              # (n_patches,)
    rectangles = f['rectangles'][:]     # (n_patches, 4, 3)
    lat        = f['lat'][:]            # (Ny, Nx)
    lon        = f['lon'][:]            # (Ny, Nx)

# 2) Reconstruct `patches` list (same fields you use)
patches = []
for rect in rectangles:
    xtlc, ytlc, dtlc = rect[0]
    length = np.linalg.norm(rect[1][:2] - rect[0][:2])
    width  = np.linalg.norm(rect[3][:2] - rect[0][:2])
    dx, dy = rect[1][0] - rect[0][0], rect[1][1] - rect[0][1]
    strike = np.degrees(np.arctan2(dy, dx))
    dz = rect[3][2] - rect[0][2]
    horiz = np.linalg.norm(rect[3][:2] - rect[0][:2])
    dip    = np.degrees(np.arctan2(-dz, horiz))
    patches.append({'xtlc': xtlc, 'ytlc': ytlc, 'dtlc': dtlc,
                    'length': length, 'width': width,
                    'strike': strike, 'dip': dip})
print(f"Loaded {len(patches)} patches from {h5_used}")


Loading: model3b.h5
Loaded 574 patches from model3b.h5


In [4]:

# 3) Project lat/lon -> X/Y (names preserved for consistency; not used for station-only compute)
wgs84    = Proj('epsg:4326')
utm_proj = Proj(proj='utm', zone=10, northern=True, ellps='WGS84')
lon_f = lon.flatten()
lat_f = lat.flatten()
x_f, y_f = transform(wgs84, utm_proj, lon_f, lat_f)
X = x_f.reshape(lat.shape)
Y = y_f.reshape(lat.shape)
print("X/Y shapes:", X.shape, Y.shape)


/tmp/ipykernel_1267125/1099165534.py:6: FutureWarning: This function is deprecated. See: https://pyproj4.github.io/pyproj/stable/gotchas.html#upgrading-to-pyproj-2-from-pyproj-1
  x_f, y_f = transform(wgs84, utm_proj, lon_f, lat_f)


X/Y shapes: (1152, 1021) (1152, 1021)


In [5]:

# 4) `stations` — reuse if provided earlier, else define 2501–2504 defaults
try:
    stations  # type: ignore
    print("Using existing `stations` dict")
except NameError:
    stations = {
        '2501': (45.95480372, -130.0090668),
        '2502': (45.96257971, -129.9910622),
        '2503': (45.94700845, -130.0265777),
        '2504': (45.95882833, -130.0114997),
        'NodeF':(45.95485,-130.008772),
        'NodeE':(45.939888,-129.974113),
        'NodeB':(45.933585,-130.013857),
    }
    print("Defined default `stations`")

# 5) station_utm (E,N) — same name as your notebook
station_utm = {}
for fid, (lat_s, lon_s) in stations.items():
    e, n, _, _ = utm.from_latlon(lat_s, lon_s)
    station_utm[fid] = (e, n)
print("station_utm:", station_utm)


Defined default `stations`
station_utm: {'2501': (421802.15868199023, 5089520.875157327), '2502': (423208.1678999553, 5090367.323655002), '2503': (420433.9977720107, 5088672.108341966), '2504': (421619.2962082646, 5089970.421139045), 'NodeF': (421825.0692740191, 5089525.727926807), 'NodeE': (424490.6507644805, 5087829.959733143), 'NodeB': (421400.9591348714, 5087168.075465049)}


In [6]:

# 6) Station-only Okada eval (fast path) -> fills your `Ustat`
fids = list(station_utm.keys())
e_s  = np.array([station_utm[f][0] for f in fids])
n_s  = np.array([station_utm[f][1] for f in fids])

Ux_s = np.zeros_like(e_s, dtype=np.float32)
Uy_s = np.zeros_like(e_s, dtype=np.float32)
Uz_s = np.zeros_like(e_s, dtype=np.float32)

for j, p in enumerate(patches):
    ux, uy, uz = OKADA(
        e_s, n_s,
        p['xtlc'], p['ytlc'], -p['dtlc'],
        p['length'], p['width'],
        p['strike'], p['dip'],
        0.0, 0.0, float(m[j]), 'R', 0.25
    )
    Ux_s += ux.astype(np.float32, copy=False)
    Uy_s += uy.astype(np.float32, copy=False)
    Uz_s += uz.astype(np.float32, copy=False)

Ustat = {fid: np.array([Ux_s[i], Uy_s[i], Uz_s[i]], dtype=np.float32) for i, fid in enumerate(fids)}
print("Filled Ustat for stations:", list(Ustat.keys()))


Filled Ustat for stations: ['2501', '2502', '2503', '2504', 'NodeF', 'NodeE', 'NodeB']


In [8]:
stations

{'2501': (45.95480372, -130.0090668),
 '2502': (45.96257971, -129.9910622),
 '2503': (45.94700845, -130.0265777),
 '2504': (45.95882833, -130.0114997),
 'NodeF': (45.95485, -130.008772),
 'NodeE': (45.939888, -129.974113),
 'NodeB': (45.933585, -130.013857)}

In [9]:
# 4) Station-only forward model: fills Ustat (Ux,Uy,Uz at each station)
fids = list(station_utm.keys())
e_s  = np.array([station_utm[f][0] for f in fids])
n_s  = np.array([station_utm[f][1] for f in fids])

Ux_s = np.zeros_like(e_s, dtype=np.float32)
Uy_s = np.zeros_like(e_s, dtype=np.float32)
Uz_s = np.zeros_like(e_s, dtype=np.float32)

for j, p in enumerate(patches):
    ux, uy, uz = OKADA(
        e_s, n_s,
        p['xtlc'], p['ytlc'], -p['dtlc'],
        p['length'], p['width'],
        p['strike'], p['dip'],
        0.0, 0.0, float(m[j]), 'R', 0.25
    )
    Ux_s += ux.astype(np.float32)
    Uy_s += uy.astype(np.float32)
    Uz_s += uz.astype(np.float32)

Ustat = {fid: np.array([Ux_s[i], Uy_s[i], Uz_s[i]], dtype=np.float32) for i, fid in enumerate(fids)}
print("Ustat:")
for k,v in Ustat.items():
    print(k, v)


Ustat:
2501 [ 0.05418807 -0.10779496  1.360627  ]
2502 [0.33146197 0.08988964 0.7676518 ]
2503 [-0.32557115 -0.21207467  0.53367966]
2504 [-0.00809423 -0.06251713  1.4313295 ]
NodeF [ 0.06463502 -0.10393368  1.3589898 ]
NodeE [ 0.3032645  -0.00416223  0.5507906 ]
NodeB [-0.2566781  -0.2244109   0.64089787]


In [7]:

# 7) Baseline projection helper — same semantics
def predict_range_change(fidA, fidB):
    eA, nA = station_utm[fidA]
    eB, nB = station_utm[fidB]
    baseline = np.array([eB-eA, nB-nA], dtype=np.float64)
    L = np.hypot(baseline[0], baseline[1])
    if L == 0:
        return 0.0
    u_hat = baseline / L
    dU = Ustat[fidB][:2] - Ustat[fidA][:2]  # Ux,Uy only
    return float(np.dot(dU, u_hat))

for A, B in [('2503','2504'),('2502','2504'),('2502','2503')]:
    print(f"Predicted Δrange {A}→{B}: {predict_range_change(A,B):.4f} m")


Predicted Δrange 2503→2504: 0.3245 m
Predicted Δrange 2502→2504: 0.3664 m
Predicted Δrange 2502→2503: 0.7181 m



### Optional: Save outputs (kept off by default)
Uncomment if you want files; otherwise everything stays in-memory with the same variable names.


In [ ]:

# np.savez('Ustat_3b_station_only.npz', **{k: Ustat[k] for k in Ustat})
# print("Saved Ustat_3b_station_only.npz")
